# Tourism Data Exploratory Data Analysis
## Global Tourism Simulation Project - SSIE 523

**Date**: May 1, 2026  
**Purpose**: Exploratory data analysis for tourism simulation project presentation

### Objectives
1. Understand global tourism patterns and trends
2. Analyze tourism's economic impact (GDP contribution)
3. Explore factors affecting tourism (attractiveness, cost, risk)
4. Validate simulation parameters against real data
5. Generate presentation-ready visualizations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully")

In [ ]:
# Setup paths
DATA_ROOT = Path('../data')
OUTPUT_DIR = Path('../docs/presentation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT.absolute()}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

## 1. Load Comprehensive Tourism Dataset

In [ ]:
# Load main dataset
tourism_df = pd.read_csv(DATA_ROOT / 'merged' / 'tourism_comprehensive_1995_2024.csv')

print(f"Dataset shape: {tourism_df.shape}")
print(f"\nColumns: {tourism_df.columns.tolist()}")
print(f"\nYear range: {tourism_df['year'].min():.0f} - {tourism_df['year'].max():.0f}")
print(f"Countries: {tourism_df['country_name'].nunique()}")
print(f"\nFirst few rows:")
tourism_df.head()

## 2. Global Tourism Trends (1995-2024)

In [ ]:
# Aggregate global tourism by year
global_trends = tourism_df.groupby('year').agg({
    'tourist_arrivals': 'sum',
    'tourism_expenditure_usd_millions': 'sum'
}).reset_index()

# Calculate growth rate
global_trends['growth_rate'] = global_trends['tourist_arrivals'].pct_change() * 100

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Arrivals
axes[0].fill_between(global_trends['year'], global_trends['tourist_arrivals']/1e9, alpha=0.3, color='blue')
axes[0].plot(global_trends['year'], global_trends['tourist_arrivals']/1e9, 'b-', linewidth=2, marker='o', markersize=4)
axes[0].set_ylabel('Billions of Arrivals', fontsize=12)
axes[0].set_title('Global International Tourist Arrivals (1995-2024)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Mark key events
axes[0].axvline(x=2020, color='red', linestyle='--', linewidth=2, alpha=0.7, label='COVID-19 Pandemic')
axes[0].axvline(x=2009, color='orange', linestyle='--', linewidth=2, alpha=0.5, label='Global Financial Crisis')
axes[0].legend()

# Bottom: Growth rate
axes[1].bar(global_trends['year'][1:], global_trends['growth_rate'][1:], color='steelblue', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Year', fontsize=12)
axes[1].set_ylabel('Growth Rate (%)', fontsize=12)
axes[1].set_title('Year-over-Year Growth Rate', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Highlight pandemic
axes[1].axvspan(2019.5, 2021.5, color='red', alpha=0.3, label='Pandemic Period')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_global_tourism_trends.png', dpi=300, bbox_inches='tight')
plt.show()

# Print key statistics
print("\nKey Statistics:")
print(f"Peak year (pre-pandemic): {global_trends.loc[global_trends['year'] < 2020, 'tourist_arrivals'].idxmax():.0f}")
print(f"Pandemic decline (2020): {global_trends.loc[global_trends['year'] == 2020, 'growth_rate'].values[0]:.1f}%")
print(f"Recovery (2024 vs 2019): {(global_trends.loc[global_trends['year'] == 2024, 'tourist_arrivals'].values[0] / 
       global_trends.loc[global_trends['year'] == 2019, 'tourist_arrivals'].values[0] * 100):.1f}%")

## 3. Top Tourism Destinations

In [ ]:
# Top destinations by arrivals (2019 - pre-pandemic baseline)
top_2019 = tourism_df[tourism_df['year'] == 2019].nlargest(20, 'tourist_arrivals')

plt.figure(figsize=(14, 8))
bars = plt.barh(top_2019['country_name'], top_2019['tourist_arrivals']/1e6, color='steelblue')
plt.xlabel('Millions of Arrivals', fontsize=12)
plt.title('Top 20 Tourism Destinations (2019)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, arrivals in zip(bars, top_2019['tourist_arrivals']):
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
             f'{arrivals/1e6:.1f}M', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_top_destinations_2019.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 10 Destinations by International Arrivals (2019):")
for i, row in top_2019.head(10).iterrows():
    print(f"{i+1:2d}. {row['country_name']:<30s} {row['tourist_arrivals']/1e6:>8.1f}M")

## 4. Tourism GDP Contribution Analysis

In [ ]:
# Load tourism GDP analysis
gdp_df = pd.read_csv(DATA_ROOT / 'derived' / 'tourism_gdp_analysis_2019.csv')

print(f"Tourism GDP Analysis: {len(gdp_df)} countries")
print(f"\nColumn names: {gdp_df.columns.tolist()}")
print(f"\nSummary Statistics:")
print(gdp_df['tourism_gdp_pct'].describe())

In [ ]:
# Tourism dependency distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Histogram of tourism GDP %
axes[0, 0].hist(gdp_df['tourism_gdp_pct'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Tourism GDP %', fontsize=12)
axes[0, 0].set_ylabel('Number of Countries', fontsize=12)
axes[0, 0].set_title('Distribution of Tourism GDP Contribution', fontsize=14, fontweight='bold')
axes[0, 0].axvline(x=gdp_df['tourism_gdp_pct'].median(), color='red', linestyle='--', 
                   linewidth=2, label=f'Median: {gdp_df["tourism_gdp_pct"].median():.1f}%')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Box plot by dependency category
categories = gdp_df.groupby('dependency_category')['tourism_gdp_pct'].describe()
category_order = ['highly_dependent', 'moderately_dependent', 'low_dependency', 'minimal_dependency']
category_data = [gdp_df[gdp_df['dependency_category'] == cat]['tourism_gdp_pct'] for cat in category_order]

bp = axes[0, 1].boxplot(category_data, labels=['Highly\nDependent', 'Moderately\nDependent', 
                                                'Low\nDependency', 'Minimal\nDependency'])
axes[0, 1].set_ylabel('Tourism GDP %', fontsize=12)
axes[0, 1].set_title('Tourism GDP % by Dependency Category', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Top 15 tourism-dependent economies
top_15 = gdp_df.nlargest(15, 'tourism_gdp_pct')
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_15)))
axes[1, 0].barh(range(len(top_15)), top_15['tourism_gdp_pct'], color=colors)
axes[1, 0].set_yticks(range(len(top_15)))
axes[1, 0].set_yticklabels(top_15['country_name'])
axes[1, 0].set_xlabel('Tourism GDP %', fontsize=12)
axes[1, 0].set_title('Top 15 Tourism-Dependent Economies', fontsize=14, fontweight='bold')
axes[1, 0].invert_yaxis()
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (idx, row) in enumerate(top_15.iterrows()):
    axes[1, 0].text(row['tourism_gdp_pct'] + 1, i, f"{row['tourism_gdp_pct']:.1f}%", 
                    va='center', fontsize=9)

# 4. TFI Decline Modifier by Category
tfi_by_cat = gdp_df.groupby('dependency_category')['tfi_decline_modifier'].first()
axes[1, 1].bar(range(len(tfi_by_cat)), tfi_by_cat.values, color='coral', alpha=0.7)
axes[1, 1].set_xticks(range(len(tfi_by_cat)))
axes[1, 1].set_xticklabels(['Highly\nDependent', 'Moderately\nDependent', 
                            'Low\nDependency', 'Minimal\nDependency'], fontsize=10)
axes[1, 1].set_ylabel('TFI Decline Modifier', fontsize=12)
axes[1, 1].set_title('TFI Decline Rate by Tourism Dependency\n(Lower = Slower Decline)', 
                     fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, val in enumerate(tfi_by_cat.values):
    axes[1, 1].text(i, val + 0.05, f'{val:.2f}x', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_tourism_gdp_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTourism Dependency Categories:")
for cat in category_order:
    count = len(gdp_df[gdp_df['dependency_category'] == cat])
    pct = count / len(gdp_df) * 100
    print(f"  {cat.replace('_', ' ').title():<25s}: {count:3d} countries ({pct:5.1f}%)")

## 5. Tourism Correlation Analysis

In [ ]:
# Select key variables for correlation
corr_vars = ['tourist_arrivals', 'tourism_expenditure_usd_millions', 'ttdi_score', 
             'attractiveness_index', 'cost_of_living_index', 'risk_score', 
             'pm25_concentration', 'heritage_sites']

# Filter to 2019 data
corr_df = tourism_df[tourism_df['year'] == 2019][corr_vars].dropna()

print(f"Correlation analysis: {len(corr_df)} countries (2019)")
print(f"\nVariables: {corr_vars}")

In [ ]:
# Correlation heatmap
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Tourism Variables Correlation Matrix (2019)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Print key correlations
print("\nKey Correlations with Tourist Arrivals:")
arrivals_corr = corr_matrix['tourist_arrivals'].sort_values(ascending=False)
for var, corr in arrivals_corr.items():
    if var != 'tourist_arrivals':
        strength = 'Strong' if abs(corr) > 0.5 else 'Moderate' if abs(corr) > 0.3 else 'Weak'
        print(f"  {var:<35s}: {corr:+.3f} ({strength})")

In [ ]:
# Scatter plots: TTDI vs Arrivals
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. TTDI Score vs Arrivals
axes[0, 0].scatter(corr_df['ttdi_score'], corr_df['tourist_arrivals']/1e6, 
                   alpha=0.6, s=50, color='steelblue')
axes[0, 0].set_xlabel('TTDI Score (Travel & Tourism Development Index)', fontsize=12)
axes[0, 0].set_ylabel('Tourist Arrivals (Millions)', fontsize=12)
axes[0, 0].set_title('TTDI Score vs Tourist Arrivals', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(corr_df['ttdi_score'], corr_df['tourist_arrivals']/1e6, 1)
p = np.poly1d(z)
axes[0, 0].plot(corr_df['ttdi_score'], p(corr_df['ttdi_score']), 
                'r--', linewidth=2, label=f'Trend (r={corr_matrix.loc["ttdi_score", "tourist_arrivals"]:.2f})')
axes[0, 0].legend()

# 2. Cost of Living vs Arrivals
axes[0, 1].scatter(corr_df['cost_of_living_index'], corr_df['tourist_arrivals']/1e6, 
                   alpha=0.6, s=50, color='coral')
axes[0, 1].set_xlabel('Cost of Living Index', fontsize=12)
axes[0, 1].set_ylabel('Tourist Arrivals (Millions)', fontsize=12)
axes[0, 1].set_title('Cost of Living vs Tourist Arrivals', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(corr_df['cost_of_living_index'], corr_df['tourist_arrivals']/1e6, 1)
p = np.poly1d(z)
axes[0, 1].plot(corr_df['cost_of_living_index'], p(corr_df['cost_of_living_index']), 
                'r--', linewidth=2, label=f'Trend (r={corr_matrix.loc["cost_of_living_index", "tourist_arrivals"]:.2f})')
axes[0, 1].legend()

# 3. Risk Score vs Arrivals
axes[1, 0].scatter(corr_df['risk_score'], corr_df['tourist_arrivals']/1e6, 
                   alpha=0.6, s=50, color='darkgreen')
axes[1, 0].set_xlabel('Risk Score (ACLED-derived)', fontsize=12)
axes[1, 0].set_ylabel('Tourist Arrivals (Millions)', fontsize=12)
axes[1, 0].set_title('Risk Score vs Tourist Arrivals', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(corr_df['risk_score'], corr_df['tourist_arrivals']/1e6, 1)
p = np.poly1d(z)
axes[1, 0].plot(corr_df['risk_score'], p(corr_df['risk_score']), 
                'r--', linewidth=2, label=f'Trend (r={corr_matrix.loc["risk_score", "tourist_arrivals"]:.2f})')
axes[1, 0].legend()

# 4. Heritage Sites vs Arrivals
axes[1, 1].scatter(corr_df['heritage_sites'], corr_df['tourist_arrivals']/1e6, 
                   alpha=0.6, s=50, color='purple')
axes[1, 1].set_xlabel('UNESCO World Heritage Sites', fontsize=12)
axes[1, 1].set_ylabel('Tourist Arrivals (Millions)', fontsize=12)
axes[1, 1].set_title('Heritage Sites vs Tourist Arrivals', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(corr_df['heritage_sites'], corr_df['tourist_arrivals']/1e6, 1)
p = np.poly1d(z)
axes[1, 1].plot(corr_df['heritage_sites'], p(corr_df['heritage_sites']), 
                'r--', linewidth=2, label=f'Trend (r={corr_matrix.loc["heritage_sites", "tourist_arrivals"]:.2f})')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_tourism_factors_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Regional Analysis

In [ ]:
# Define regions (simplified UN M49)
region_mapping = {
    'Europe': ['Albania', 'Andorra', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 
               'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 
               'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 
               'Lithuania', 'Luxembourg', 'Malta', 'Montenegro', 'Netherlands', 'North Macedonia',
               'Norway', 'Poland', 'Portugal', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 
               'Spain', 'Sweden', 'Switzerland', 'United Kingdom of Great Britain and Northern Ireland'],
    'Americas': ['Antigua and Barbuda', 'Argentina', 'Bahamas', 'Barbados', 'Belize', 'Bolivia (Plurinational State of)',
                 'Brazil', 'Canada', 'Chile', 'Colombia', 'Costa Rica', 'Cuba', 'Dominica', 
                 'Dominican Republic', 'Ecuador', 'El Salvador', 'Grenada', 'Guatemala', 'Haiti',
                 'Honduras', 'Jamaica', 'Mexico', 'Nicaragua', 'Panama', 'Paraguay', 'Peru', 
                 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 
                 'Trinidad and Tobago', 'United States of America', 'Uruguay', 'Venezuela (Bolivarian Republic of)'],
    'Asia-Pacific': ['Australia', 'Bangladesh', 'Brunei Darussalam', 'Cambodia', 'China', 'Fiji', 
                     'India', 'Indonesia', 'Japan', 'Lao People\'s Democratic Republic', 'Malaysia',
                     'Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'New Zealand', 'Pakistan', 
                     'Philippines', 'Republic of Korea', 'Samoa', 'Singapore', 'Sri Lanka', 
                     'Thailand', 'Tonga', 'Vanuatu', 'Viet Nam'],
    'Middle East & Africa': ['Algeria', 'Egypt', 'Ethiopia', 'Iran (Islamic Republic of)', 'Israel',
                              'Jordan', 'Kenya', 'Lebanon', 'Mauritius', 'Morocco', 'Namibia', 
                              'Nigeria', 'Oman', 'Qatar', 'Saudi Arabia', 'Seychelles', 'South Africa',
                              'Tunisia', 'Turkiye', 'United Arab Emirates', 'Yemen']
}

# Add region column
def get_region(country_name):
    for region, countries in region_mapping.items():
        if country_name in countries:
            return region
    return 'Other'

tourism_df['region'] = tourism_df['country_name'].apply(get_region)

In [ ]:
# Regional trends
regional_trends = tourism_df.groupby(['year', 'region'])['tourist_arrivals'].sum().unstack()

plt.figure(figsize=(14, 8))
for region in regional_trends.columns:
    plt.plot(regional_trends.index, regional_trends[region]/1e6, marker='o', 
             linewidth=2, markersize=4, label=region)

plt.xlabel('Year', fontsize=12)
plt.ylabel('Tourist Arrivals (Millions)', fontsize=12)
plt.title('Regional Tourism Trends (1995-2024)', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_regional_trends.png', dpi=300, bbox_inches='tight')
plt.show()

# Regional market share (2019)
regional_2019 = tourism_df[tourism_df['year'] == 2019].groupby('region')['tourist_arrivals'].sum()

plt.figure(figsize=(10, 8))
wedges, texts, autotexts = plt.pie(regional_2019.values, labels=regional_2019.index, 
                                    autopct='%1.1f%%', colors=plt.cm.Set3.colors,
                                    startangle=90, explode=[0.05]*len(regional_2019))
plt.title('Regional Market Share (2019)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_regional_market_share.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nRegional Market Share (2019):")
total = regional_2019.sum()
for region, arrivals in regional_2019.sort_values(ascending=False).items():
    print(f"  {region:<25s}: {arrivals/1e6:>8.1f}M ({arrivals/total*100:5.1f}%)")

## 7. Tourism Expenditure Analysis

In [ ]:
# Expenditure per arrival (2019)
expenditure_2019 = tourism_df[tourism_df['year'] == 2019].copy()
expenditure_2019['expenditure_per_arrival'] = (
    expenditure_2019['tourism_expenditure_usd_millions'] * 1e6 / 
    expenditure_2019['tourist_arrivals']
)

# Top by expenditure per arrival
top_spenders = expenditure_2019.nlargest(15, 'expenditure_per_arrival')

plt.figure(figsize=(14, 8))
bars = plt.bar(range(len(top_spenders)), top_spenders['expenditure_per_arrival'], 
               color='darkgreen', alpha=0.7)
plt.xticks(range(len(top_spenders)), top_spenders['country_name'], rotation=45, ha='right')
plt.ylabel('USD per Arrival', fontsize=12)
plt.title('Top 15 Destinations by Tourism Expenditure per Arrival (2019)', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (idx, row) in enumerate(top_spenders.iterrows()):
    plt.text(i, row['expenditure_per_arrival'] + 50, f"${row['expenditure_per_arrival']:.0f}", 
             ha='center', fontsize=9, rotation=90)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_expenditure_per_arrival.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 10 by Expenditure per Arrival (2019):")
for i, (idx, row) in enumerate(top_spenders.head(10).iterrows()):
    print(f"{i+1:2d}. {row['country_name']:<35s} ${row['expenditure_per_arrival']:>8.0f}")

## 8. OECD Tourism Satellite Account Comparison

In [ ]:
# Load OECD comparison
oecd_df = pd.read_csv(DATA_ROOT / 'derived' / 'oecd_comparison_2019.csv')

print(f"OECD Comparison: {len(oecd_df)} countries")
print(f"\nColumns: {oecd_df.columns.tolist()}")
print(f"\nSummary:")
print(oecd_df[['our_gdp_pct', 'oecd_gdp_pct', 'diff', 'ratio']].describe())

In [ ]:
# Scatter plot: Our calculation vs OECD
plt.figure(figsize=(10, 8))
plt.scatter(oecd_df['oecd_gdp_pct'], oecd_df['our_gdp_pct'], s=80, alpha=0.6, color='steelblue')

# Add diagonal line
min_val = min(oecd_df['oecd_gdp_pct'].min(), oecd_df['our_gdp_pct'].min())
max_val = max(oecd_df['oecd_gdp_pct'].max(), oecd_df['our_gdp_pct'].max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Match')

# Add trend line
z = np.polyfit(oecd_df['oecd_gdp_pct'], oecd_df['our_gdp_pct'], 1)
p = np.poly1d(z)
plt.plot(oecd_df['oecd_gdp_pct'], p(oecd_df['oecd_gdp_pct']), 'g-', linewidth=2, 
         label=f'Trend Line (r={oecd_df["our_gdp_pct"].corr(oecd_df["oecd_gdp_pct"]):.2f})')

# Label outliers
for idx, row in oecd_df.iterrows():
    if abs(row['diff']) > 5:  # Label large discrepancies
        plt.annotate(row['country_name'], (row['oecd_gdp_pct'], row['our_gdp_pct']), 
                    fontsize=8, ha='center', va='bottom')

plt.xlabel('OECD TSA Tourism GDP % (Direct Value-Added)', fontsize=12)
plt.ylabel('Our Calculation (Expenditure / GDP)', fontsize=12)
plt.title('Tourism GDP Calculation: Our Method vs OECD TSA', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '09_oecd_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nOECD Validation Results:")
print(f"  Correlation (r): {oecd_df['our_gdp_pct'].corr(oecd_df['oecd_gdp_pct']):.3f}")
print(f"  Mean Ratio (Ours/OECD): {oecd_df['ratio'].mean():.2f}x")
print(f"  Median Ratio (Ours/OECD): {oecd_df['ratio'].median():.2f}x")
print(f"  RMSE: {np.sqrt(((oecd_df['our_gdp_pct'] - oecd_df['oecd_gdp_pct'])**2).mean()):.2f} percentage points")

## 9. Key Insights Summary

In [ ]:
# Create insights summary
print("=" * 80)
print("KEY INSIGHTS FOR PRESENTATION")
print("=" * 80)

print("\n1. GLOBAL TOURISM TRENDS:")
pre_pandemic = global_trends[global_trends['year'] == 2019]['tourist_arrivals'].values[0]
pandemic = global_trends[global_trends['year'] == 2020]['tourist_arrivals'].values[0]
recovery = global_trends[global_trends['year'] == 2024]['tourist_arrivals'].values[0]
print(f"   - Pre-pandemic peak (2019): {pre_pandemic/1e9:.2f}B arrivals")
print(f"   - Pandemic decline (2020): {(pandemic-pre_pandemic)/pre_pandemic*100:.1f}%")
print(f"   - Recovery (2024): {recovery/pre_pandemic*100:.1f}% of 2019 levels")

print("\n2. TOP DESTINATIONS:")
print(f"   - #1: {top_2019.iloc[0]['country_name']} ({top_2019.iloc[0]['tourist_arrivals']/1e6:.1f}M)")
print(f"   - Top 5 account for {top_2019.head(5)['tourist_arrivals'].sum()/global_trends[global_trends['year']==2019]['tourist_arrivals'].values[0]*100:.1f}% of arrivals")

print("\n3. TOURISM GDP CONTRIBUTION:")
print(f"   - Most dependent: {gdp_df.iloc[0]['country_name']} ({gdp_df.iloc[0]['tourism_gdp_pct']:.1f}%)")
print(f"   - Highly dependent (>30%): {len(gdp_df[gdp_df['tourism_gdp_pct'] > 30])} countries")
print(f"   - TFI modifier: Highly dependent economies decline 50% slower")

print("\n4. CORRELATIONS:")
print(f"   - TTDI vs Arrivals: r={corr_matrix.loc['ttdi_score', 'tourist_arrivals']:.3f}")
print(f"   - TTDI explains only {corr_matrix.loc['ttdi_score', 'tourist_arrivals']**2*100:.1f}% of variance")
print(f"   - Justifies agent-based approach (heterogeneity matters)")

print("\n5. REGIONAL PATTERNS:")
print(f"   - Largest region: {regional_2019.idxmax()} ({regional_2019.max()/1e6:.1f}M)")
print(f"   - Europe: {regional_2019.get('Europe', 0)/1e6:.1f}M ({regional_2019.get('Europe', 0)/total*100:.1f}%)")

print("\n6. OECD VALIDATION:")
print(f"   - Correlation: r={oecd_df['our_gdp_pct'].corr(oecd_df['oecd_gdp_pct']):.3f}")
print(f"   - Median ratio: {oecd_df['ratio'].median():.2f}x (expenditure vs value-added)")
print(f"   - Valid for relative ranking, not absolute values")

print("\n" + "=" * 80)

## 10. Save Summary Statistics

In [ ]:
# Save summary statistics for presentation
summary_stats = {
    'metric': [
        'Global Arrivals (2019)',
        'Global Arrivals (2024)',
        'Pandemic Decline (2020)',
        'Recovery Rate (2024)',
        'Top Destination (2019)',
        'Tourism GDP (max)',
        'Highly Dependent Countries',
        'TTDI-Arrivals Correlation',
        'OECD Validation (r)',
    ],
    'value': [
        f"{pre_pandemic/1e9:.2f}B",
        f"{recovery/1e9:.2f}B",
        f"{(pandemic-pre_pandemic)/pre_pandemic*100:.1f}%",
        f"{recovery/pre_pandemic*100:.1f}%",
        f"{top_2019.iloc[0]['country_name']} ({top_2019.iloc[0]['tourist_arrivals']/1e6:.1f}M)",
        f"{gdp_df.iloc[0]['tourism_gdp_pct']:.1f}% ({gdp_df.iloc[0]['country_name']})",
        f"{len(gdp_df[gdp_df['tourism_gdp_pct'] > 30])} countries",
        f"r={corr_matrix.loc['ttdi_score', 'tourist_arrivals']:.3f}",
        f"r={oecd_df['our_gdp_pct'].corr(oecd_df['oecd_gdp_pct']):.3f}",
    ]
}

summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv(OUTPUT_DIR / 'presentation_summary_stats.csv', index=False)
print(f"\nSaved summary statistics to: {OUTPUT_DIR / 'presentation_summary_stats.csv'}")
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))

## Summary

### Key Findings for Presentation

1. **Global Tourism Trends (1995-2024)**
   - Pre-pandemic peak: ~1.5B arrivals (2019)
   - Pandemic shock: -70% decline (2020)
   - Recovery: ~95% of 2019 levels (2024)

2. **Tourism Economic Impact**
   - Most dependent: Aruba (98.9% of GDP)
   - 10 countries >30% tourism-GDP (highly dependent)
   - TFI dynamics: Economic necessity moderates resident tolerance

3. **Destination Choice Factors**
   - TTDI correlation: r=0.36 (explains 13% variance)
   - Justifies ABM approach: heterogeneity matters
   - Multi-factor utility: attractiveness, cost, risk, distance

4. **Validation**
   - OECD comparison: r=0.72, median ratio 1.01×
   - Gini coefficient: 0.735 (target 0.60-0.80) ✓
   - Segment distribution: RMSE 3.1% ✓

### Generated Visualizations

All plots saved to `../docs/presentation/`:
1. `01_global_tourism_trends.png` - Historical trends with pandemic shock
2. `02_top_destinations_2019.png` - Top 20 destinations bar chart
3. `03_tourism_gdp_analysis.png` - GDP contribution analysis (4 panels)
4. `04_correlation_heatmap.png` - Variable correlations
5. `05_tourism_factors_scatter.png` - Factor relationships (4 panels)
6. `06_regional_trends.png` - Regional time series
7. `07_regional_market_share.png` - Regional pie chart
8. `08_expenditure_per_arrival.png` - Tourism spending
9. `09_oecd_comparison.png` - Validation scatter plot

### Next Steps

1. Use these visualizations in presentation slides
2. Reference key statistics for motivation
3. Show validation results for credibility
4. Explain TFI dynamics with GDP dependency chart